In [0]:
# 1. Drop bronze entirely (clears old appends + truncate commit)
spark.sql("DROP TABLE IF EXISTS retaildp.bronze.pos_rtlog")

# 2. Clear bronze Auto Loader checkpoint (confirm path against your bronze notebook)
dbutils.fs.rm("abfss://checkpoints@stretaildpsatyaki01.dfs.core.windows.net/bronze/pos_rtlog/", recurse=True)

# 3. Clear silver checkpoint again (the failed run may have re-created it)
dbutils.fs.rm("abfss://checkpoints@stretaildpsatyaki01.dfs.core.windows.net/silver/sa_store_day/", recurse=True)
spark.sql("DROP TABLE IF EXISTS retaildp.silver.sa_store_day")
spark.sql("DROP TABLE IF EXISTS retaildp.quarantine.silver_sa_store_day_rejects")
print("Reset complete.")

In [0]:
dbutils.fs.ls("abfss://raw@stretaildpsatyaki01.dfs.core.windows.net/pos/")
   # expect store=33487, store=39876, store=41203 — and NO store=1/2/3

In [0]:
spark.table("retaildp.bronze.pos_rtlog").select("store").distinct().orderBy("store").show()
   # expect 33487, 39876, 41203

In [0]:
base = "abfss://raw@stretaildpsatyaki01.dfs.core.windows.net/pos/store=33487/"
found = False
for d1 in dbutils.fs.ls(base):                 # date=
    for d2 in dbutils.fs.ls(d1.path):          # hour=
        for f in dbutils.fs.ls(d2.path):       # rtlog.ndjson
            print(f.path, f.size, "bytes")
            found = True
if not found:
    print("NO data files found under store=33487 — folders are empty.")

In [0]:
dbutils.fs.rm("abfss://checkpoints@stretaildpsatyaki01.dfs.core.windows.net/pos_rtlog/", recurse=True)
print("Real bronze checkpoint + schema cleared.")

In [0]:
spark.sql("""
    WITH dups AS (
        SELECT store, date, tran_head.register AS register, tran_head.tran_no AS tran_no
        FROM retaildp.bronze.pos_rtlog
        GROUP BY store, date, tran_head.register, tran_head.tran_no
        HAVING COUNT(*) > 1
    )
    SELECT
        b.store, b.date, b.tran_head.register, b.tran_head.tran_no,
        b.tran_head.tran_seq_no, b.tran_head.tran_datetime,
        b.tran_head.value
    FROM retaildp.bronze.pos_rtlog b
    JOIN dups d
      ON b.store = d.store AND b.date = d.date
     AND b.tran_head.register = d.register
     AND b.tran_head.tran_no = d.tran_no
    ORDER BY b.store, b.date, b.tran_head.register, b.tran_head.tran_no, b.tran_head.tran_datetime
""").show(40, truncate=False)

In [0]:
spark.sql("DROP TABLE IF EXISTS retaildp.silver.sa_tran_head")
spark.sql("DROP TABLE IF EXISTS retaildp.quarantine.silver_sa_tran_head_rejects")
dbutils.fs.rm(
    "abfss://checkpoints@stretaildpsatyaki01.dfs.core.windows.net/silver/sa_tran_head/",
    recurse=True,
)
print("sa_tran_head reset complete.")

In [0]:
spark.sql("""
WITH dups AS (
    SELECT tran_head.tran_seq_no AS seq
    FROM retaildp.bronze.pos_rtlog
    GROUP BY tran_head.tran_seq_no
    HAVING COUNT(*) > 1
)
SELECT
    b.store,
    b.date,
    b.tran_head.register,
    b.tran_head.tran_no,
    b.tran_head.tran_seq_no,
    b.tran_head.tran_datetime,
    b.tran_head.tran_type,
    b.tran_head.value
FROM retaildp.bronze.pos_rtlog b
JOIN dups d ON b.tran_head.tran_seq_no = d.seq
ORDER BY b.tran_head.tran_seq_no, b.tran_head.tran_datetime
""").show(40, truncate=False)